# S02 · MLA weight absorption on the course's MLA module (the idea behind FlashMLA)

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 5. Sources: DeepSeek-V2 (arXiv:2405.04434); github.com/deepseek-ai/FlashMLA (13k stars, MIT).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Take the course's `MultiHeadLatentAttention` (which up-projects its whole latent cache to K and V
at every step) and compute the *same* outputs in absorbed form from the same weights, never
building K or V. Then count what absorption saves.

## Theory and equations
$q_i^\top W_{UK}^i c_t = (W_{UK}^{i\top} q_i)^\top c_t$ and $o_i = W_{UV}^i \sum_t p_t c_t$; optionally fold $W_O^i W_{UV}^i$.
The cache stays $c_t \in \mathbb{R}^{d_c}$ plus the shared RoPE key $k^R_t \in \mathbb{R}^{d_R}$.

## Diagram
```
course module (explicit):  latent (T x d_c) --k_up_proj--> K (H x T x d_h) --v_up_proj--> V ... SDPA
absorbed (this notebook):  q_i --W_UK^T--> q~_i (d_c) . latent -> softmax -> attend latent -> W_UV (or W_O W_UV)
```

## Implementation

In [ ]:
from kv_cache_variants.attention.mla import MultiHeadLatentAttention
from kv_cache_variants.rope import build_rope_cache
from inference_lab import mla

torch.manual_seed(0)
m = MultiHeadLatentAttention(d_model=256, num_heads=8, latent_dim=64, rope_head_dim=16)
rope = build_rope_cache(16, max_seq_len=16384)
x = torch.randn(1, 32, 256)
with torch.no_grad():
    ref, cache = m(x, rope=rope)                                # the course's forward (explicit)
    out, cache2 = mla.absorbed_forward(m, x, rope=rope)         # absorbed, same weights
    step = torch.randn(1, 1, 256)
    ref1, _ = m(step, past_kv=cache, rope=rope)
    out1, _ = mla.absorbed_forward(m, step, past_kv=cache2, rope=rope, fold_output=True)
print("prefill max |diff|:", float((out - ref).abs().max()), " decode max |diff|:", float((out1 - ref1).abs().max()))

## Experiment: decode FLOPs vs context (*theoretical*, from the module's shapes)

In [ ]:
Ts = [256, 1024, 4096, 16384, 65536]
fe = [mla.decode_flops(m, T, absorbed=False) for T in Ts]
fa = [mla.decode_flops(m, T, absorbed=True) for T in Ts]
for T, a, b in zip(Ts, fe, fa):
    print(f"T={T:6d}  explicit {a/1e6:9.1f} MFLOP   absorbed {b/1e6:7.1f} MFLOP   {a/b:5.1f}x")
plt.loglog(Ts, fe, "o-", color=C["waste"], label="explicit: up-project the cache")
plt.loglog(Ts, fa, "o-", color=C["compute"], label="absorbed")
plt.xlabel("context T"); plt.ylabel("attention FLOPs per decode step"); plt.legend(); plt.show()

## Optional: wall-clock on your machine (*measured* only when you run it)

In [ ]:
import time
def bench(f, T, reps=7):
    c = (torch.randn(1, T, 64), torch.randn(1, T, 16))
    best = float("inf")
    with torch.no_grad():
        f(step, c)
        for _ in range(reps):
            t = time.perf_counter(); f(step, c); best = min(best, time.perf_counter() - t)
    return best
for T in (1024, 8192):
    te = bench(lambda s, c: m(s, past_kv=c, rope=rope), T)
    ta = bench(lambda s, c: mla.absorbed_forward(m, s, past_kv=c, rope=rope), T)
    print(f"T={T}: explicit {1e3*te:.2f} ms, absorbed {1e3*ta:.2f} ms")

## Interpretation
Both forms agree to float precision. The explicit form's cost grows with $T \cdot H \cdot d_h
\cdot d_c$ (re-up-projecting the cache every step); the absorbed form's with $T \cdot H \cdot d_c$.

## Limitations
CPU PyTorch. On GPU the absorbed decode becomes compute-bound, which kernels like FlashMLA target.

## Conclusion
Decompression moves onto the single query. It is never applied to the cache.